## Azure AI Evaluation Example

This notebook demonstrates how to configure and use the Azure AI Evaluation SDK to evaluate an agent using a critic agent. The setup includes loading environment variables, configuring the model, and running both synchronous and asynchronous evaluations.

- **Environment Setup:** Loads credentials and endpoints from a `.env` file.
- **Model Configuration:** Sets up the Azure OpenAI model configuration.
- **Critic Agent:** Instantiates a critic agent for evaluation tasks.
- **Evaluation:** Runs both standard and auto evaluation methods on the specified agent and project.

Refer to the code cells above for implementation details.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle
from pprint import pprint
from azure.ai.evaluation import AzureOpenAIModelConfiguration, AzureAIProject
# from azure.ai.evaluation._agents._critic_agent import CriticAgent
from _critic_agent import CriticAgent

from dotenv import load_dotenv


In [ ]:
load_dotenv()

In [ ]:
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT"),
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.environ.get("AZURE_OPENAI_DEPLOYMENT"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-02-01")
)
#
critic_agent = CriticAgent(model_config=model_config)

In [ ]:
azure_ai_project = {
    "azure_endpoint": os.environ.get("PROJECT_ENDPOINT"),
}


### Test Fetching Agents and Threads

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation._converters._ai_services import AIAgentConverter

project_client = AIProjectClient(
    endpoint=azure_ai_project.get("azure_endpoint"),
    credential=DefaultAzureCredential(),
)
threads = project_client.agents.threads.list()




In [ ]:
weather_agent = project_client.agents.get_agent(agent_id="asst_2TiZjcHhE4i0CSwaln0ab085")
weather_agent

In [ ]:
threads.next().id

In [ ]:


converter = AIAgentConverter(project_client)
conversation = converter.prepare_evaluation_data(thread_ids="thread_UXWZTzYmTIM3IMtNh4xSm2Mv")
conversation

### Streamlining

Evaluates an Agent/thread against the provided evaluators

In [ ]:
results = critic_agent.evaluate(thread_id="thread_IO9oEfO11j2uc4IUcY2u6jqS", azure_ai_project=azure_ai_project, evaluators=['IntentResolution'])

In [ ]:
results

In [ ]:
results = critic_agent.evaluate(agent_id="asst_2TiZjcHhE4i0CSwaln0ab085", azure_ai_project=azure_ai_project, evaluators=[ 'IntentResolution'], max_threads=2)

### Routing
1) Auto select the most relevant evaluator for each thread of an given agent Agent
2) Auto Select the most relevant evaluator for a given thread

In [ ]:

import nest_asyncio
import asyncio

nest_asyncio.apply()

critic_agent.auto_evaluate(agent_id="asst_8LTtd9HiRi1tABmOIergHYPX", azure_ai_project=azure_ai_project)

In [ ]:
critic_agent.auto_evaluate(thread_id="thread_HJbWlH063heMgE297bstenx6", azure_ai_project=azure_ai_project)

## Analysis
The goal is to identify key error patterns. User can then update various parts of the Agent to improve the results.
- Create a summary report
- Consolidate common error types
- Make a visualization of the error clusters
- Create interactive drill down scenario for better investigating the common errors


### Step 1: Run Agent Evaluation
We'll start by evaluating an agent with multiple evaluators to generate evaluation results that we can analyze for errors and patterns.

In [ ]:
results = critic_agent.evaluate(agent_id="asst_2TiZjcHhE4i0CSwaln0ab085", azure_ai_project=azure_ai_project, evaluators=['IntentResolution', 'TaskAdherence', 'ToolCallAccuracy'], max_threads=1)


For this analysis, we'll load pre-existing evaluation results from a pickle file to demonstrate the error analysis capabilities.

In [ ]:
# import pickle
# with open("agent_evaluate_results.pkl","wb") as f:
#     pickle.dump(results, f)

# agent_evaluate_results = results   

In [ ]:
import pickle
with open("data/agent_evaluate_results_weather.pkl", "rb") as f:
    agent_evaluate_results = pickle.load(f)

Let's examine the structure of our evaluation results to understand what data we're working with.

In [ ]:
pprint(agent_evaluate_results[0]['results'])

### Step 2: Perform Error Analysis
Now we'll use the critic agent's built-in error analysis functionality to identify patterns in failed evaluations and generate insights using LLM analysis.

In [ ]:
report  = critic_agent.analyze_errors(
    evaluation_results=agent_evaluate_results,
    fails_only=True,
    num_clusters=5,
    use_llm_analysis=True,
)

#### Examine Analysis Report Structure
Let's explore what information the error analysis report contains.

In [ ]:
report.keys()

#### Error Analysis Report Structure

Based on the actual analysis report generated, here's the real structure with 7 top-level fields:

```
report/
├── summary/                     # Statistical overview (dict with 6 keys)
│   ├── total_evaluations        # Total number of evaluations analyzed
│   ├── failed_evaluations       # Count of failed evaluations
│   ├── success_rate            # Overall success percentage
│   └── ... (3 more summary stats)
│
├── patterns/                    # Failure pattern analysis (dict with 4 keys)
│   ├── evaluation_types         # Breakdown by evaluator type
│   ├── reasons                 # Common failure reasons
│   ├── total_issues            # Issue count summary
│   └── total_threads           # Thread-level statistics
│
├── error_tags/                  # List of error classifications (36 items)
│   └── [...categorical tags for error types...]
│
├── error_clusters/             # Clustering analysis results (dict with 4 keys)
│   ├── clusters                # Individual cluster details
│   ├── total_errors           # Total errors processed
│   ├── total_assigned         # Errors successfully clustered
│   └── coverage               # Clustering coverage percentage
│
├── reason_mapping/             # Error reason categorization (dict with 2 keys)
│   ├── reasons_to_names       # Mapping of reason codes to descriptions
│   └── reasons_to_eval_types  # Mapping of reasons to evaluator types
│
├── llm_analysis/               # AI-generated insights (4,602 character string)
│   └── [Comprehensive LLM analysis with patterns, recommendations]
│
└── raw_imperfect_data/         # Raw evaluation data (list with 27 items)
    └── [...original evaluation results that had issues...]
```


### Step 3: LLM Analysis Summary
The error analysis report includes insights generated by an LLM. Let's display these findings in a readable format.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(report["llm_analysis"]))

### Step 4: Visualize Error Clusters
Create visual representations of the error patterns identified in the analysis to better understand the distribution and characteristics of failures.

In [ ]:
# Visualize the error clusters
critic_agent.visualize_errors(
    error_analysis_report=report
)

### Step 5: Interactive Error Analysis
Use the interactive error analysis feature to drill down into specific error clusters and explore individual cases in detail.

In [ ]:
# drill down into a specific error cluster
critic_agent.interactive_error_analysis(
    error_analysis_report=report
)